# 05 - Baselines and models (logistic regression vs LightGBM)

**Models** predict P(next-60-minute return > 0):
- base rate (no information);
- logistic regression with C ∈ {0.01, 0.1, 1};
- two conservative LightGBM configurations.

**Baselines for trading:** buy-and-hold, and the sign of the last 1 h or 24 h return.

**Pre-registered simplicity rule:** LightGBM is used only if its mean validation AUC beats logistic regression by more than 0.005 **and** its log-loss is lower in at least 70% of folds.

Predictive metrics (AUC, log-loss, Brier, accuracy, precision, recall) do **not** show profitability on their own; that is decided by the backtest.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
import run_pipeline as rp
from src import config
T, F = config.TABLES_DIR, config.FIGURES_DIR
SYMBOL = config.SYMBOL
# False (default): display the outputs saved by scripts/run_pipeline.py.
# True re-computes the stage. Do NOT re-run 'validation' or 'test' casually:
# validation resets the experiment log and every test run adds a TEST row.
RUN_STAGE = False
def show(name, **kw):
    display(Markdown(f"**{name}**")); display(pd.read_csv(T / name, **kw))
def fig(name):
    display(Image(filename=str(F / name)))
def results(section):
    return json.loads((config.REPORTS_DIR / "results.json").read_text()).get(section, {})

In [2]:
if RUN_STAGE:
    rp.stage_validation(SYMBOL)

In [3]:
show('validation_models.csv')

**validation_models.csv**

,model,n,base_rate,mean_p,auc,logloss,brier,accuracy,precision,recall,f1,share_predicted_up,ic_spearman,mean_fold_auc,share_folds_logloss_better,logloss_gain_vs_base,logloss_gain_p_one_sided
0,base_rate,43415,0.50759,0.509682,0.496401,0.693082,0.249968,0.507590,0.507590,1.000000,0.673379,1.000000,-0.002058,0.500000,0.0,0.000000,NaN
1,logreg|C=0.01,43415,0.50759,0.510145,0.556993,0.688283,0.247573,0.543867,0.545960,0.602124,0.572668,0.559807,0.080546,0.559772,0.9,0.004800,2.696039e-23
2,logreg|C=0.1,43415,0.50759,0.510118,0.556970,0.688298,0.247580,0.543453,0.545683,0.600581,0.571817,0.558655,0.080527,0.559614,0.9,0.004784,2.055680e-22
3,logreg|C=1.0,43415,0.50759,0.510121,0.556962,0.688302,0.247581,0.543338,0.545574,0.600535,0.571737,0.558724,0.080511,0.559596,0.9,0.004780,2.616712e-22
4,"lgbm|colsample_bytree=0.8,learning_rate=0.03,m...",43415,0.50759,0.509778,0.564914,0.687199,0.247024,0.549119,0.551675,0.596361,0.573148,0.548704,0.086943,0.566154,0.9,0.005884,7.496290e-20
5,"lgbm|colsample_bytree=0.8,learning_rate=0.02,m...",43415,0.50759,0.509795,0.562503,0.688564,0.247676,0.548497,0.550905,0.597904,0.573443,0.550893,0.082253,0.563469,0.9,0.004519,2.291429e-10


In [4]:
show('validation_baselines.csv')

**validation_baselines.csv**

,strategy,mean_fold_net_sharpe,val_sharpe,val_gross_sharpe,val_annualized_return,val_max_drawdown,val_trades_per_year,auc
0,buy_and_hold,1.239748,1.009778,1.026504,0.575747,-0.785541,15.202081,NaN
1,momentum_1h_sign,-11.395592,-10.122395,-0.611202,-0.999261,-1.000000,4682.240964,0.452859
2,momentum_24h_sign,-2.226403,-1.947129,-0.084637,-0.793292,-0.999641,921.326123,0.474922


**Interpretation:** see the matching section of `reports/final_report.md`, written from these outputs.